In [7]:
from typing_extensions import TypedDict, Literal, Annotated
from typing import List
from langgraph.types import Send
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel
from operator import add

llm = init_chat_model("openai:gpt-4o")

In [ ]:
class State(TypedDict):
    dish:str
    ingredients:list[dict]
    recipe_steps:str
    plating_instructions:str


class Ingredients(BaseModel):
    name:str
    quantity:str
    unit:str

class IngredientOutput(BaseModel):

    ingredients: List[Ingredients]

In [ ]:
def list_ingredients(state:State):
    structured_llm = llm.with_structured_output(IngredientOutput)
    response = structured_llm.invoke(f"List 5-8 ingredients needed to make {state['dish']}")
    return {"ingredients" : response.ingredients}


def create_recipe(state:State):
    response = llm.invoke(f"write a step by step cooking instruction for {state['dish']}, using these ingredients {state['ingredients']}")
    return {
        "recipe_steps": response.content
    }


def describe_plating(state:State):
    response = llm.invoke(f"Describe how to beautifully plate this dish {state['dish']} based on this recipe {state["recipe_steps"]}")
    return {"plating_instructions": response.content}



In [10]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [11]:
graph.invoke({"dish": "hummus"})

c:\Users\user\ai_agent_nomad\workflow-architectures\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=IngredientOutput(ingredie...lespoons (as needed)')]), input_type=IngredientOutput])
  return self.__pydantic_serializer__.to_python(


{'dish': 'hummus',
 'ingredients': [Ingredients(name='Chickpeas', quantity='1', unit='can (15 oz)'),
  Ingredients(name='Tahini', quantity='1/4', unit='cup'),
  Ingredients(name='Garlic cloves', quantity='2', unit='pieces'),
  Ingredients(name='Lemon juice', quantity='2', unit='tablespoons'),
  Ingredients(name='Olive oil', quantity='3', unit='tablespoons'),
  Ingredients(name='Ground cumin', quantity='1', unit='teaspoon'),
  Ingredients(name='Salt', quantity='1/2', unit='teaspoon'),
  Ingredients(name='Water', quantity='2-3', unit='tablespoons (as needed)')]}